# DataCrunch 2 — Offline Transformer Pretraining

**Run this on your own GPU / supercomputer — this notebook is NOT submitted
to CrunchDAO.** Its only job is to produce `transformer_checkpoint.pt`,
which you then upload as a resource alongside the separate submission
notebook (`competition_transformer.ipynb`).

Why split it this way: CrunchDAO's own runtime is CPU-only (16 vCPU, no
GPU, per your run details) and calls `train()` 9 times inside a shared
~10h/week compute quota. Full transformer training from scratch doesn't
fit that budget. Pretraining here — with real compute and no time
pressure — then having each CrunchDAO-side `train()` call do only a small,
cheap fine-tune on newly available data is how the two constraints get
reconciled. This split-training pattern is explicitly supported by
CrunchDAO: model files can be uploaded as resources alongside a notebook
submission, and their own FAQ describes training locally and having the
platform continue training with more data during the Out-of-Sample phase.

**⚠️ Keep the `FTTransformerRegressor` class definition below byte-for-byte
identical to the copy in the submission notebook.** The checkpoint is a
`state_dict` — it only loads into a model with the exact same architecture
that produced it.


In [ ]:
# Adjust for your CUDA version if needed -- see https://pytorch.org/get-started/locally/
%pip install crunch-cli torch --upgrade --quiet --progress-bar off

# Pulls the same competition data used by the submission notebook
!crunch setup-notebook datacrunch-2 0JiCmmP21Ca88X8TApRuDHMH --size small

In [ ]:
import os

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from scipy.stats import spearmanr

import crunch

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {DEVICE}")
if DEVICE == "cpu":
    print("WARNING: no GPU detected -- pretraining here will be slow. Check your CUDA setup.")

crunch_tools = crunch.load_notebook()

## Config & helpers

In [ ]:
RANDOM_STATE = 0
ID_COLUMNS = ["id", "moon"]
CHECKPOINT_PATH = "transformer_checkpoint.pt"

torch.manual_seed(RANDOM_STATE)
np.random.seed(RANDOM_STATE)


def get_feature_columns(df: pd.DataFrame):
    return [c for c in df.columns if c not in ID_COLUMNS and c != "target"]


def spearman(y_true, y_pred) -> float:
    corr, _ = spearmanr(y_true, y_pred)
    return 0.0 if np.isnan(corr) else corr

## Model

Feature-Tokenizer Transformer (Gorishniy et al. 2021, adapted): each of the
already-quantized (0–6) features becomes a token via a value embedding plus
a learned feature-identity embedding — identity has to be injected
explicitly since a plain transformer is permutation-invariant over its
input tokens, the same role positional embeddings play for text. A learned
`[CLS]` token is prepended; its output after the encoder feeds a small
regression head.

In [ ]:
class FTTransformerRegressor(nn.Module):
    def __init__(self, n_features: int, n_bins: int = 7, d_model: int = 64,
                 n_heads: int = 4, n_layers: int = 3, dropout: float = 0.1):
        super().__init__()
        self.n_features = n_features
        self.value_embedding = nn.Embedding(n_bins, d_model)
        self.feature_id_embedding = nn.Embedding(n_features, d_model)
        self.cls_token = nn.Parameter(torch.zeros(1, 1, d_model))
        nn.init.normal_(self.cls_token, std=0.02)

        encoder_layer = nn.TransformerEncoderLayer(
            d_model=d_model, nhead=n_heads, dim_feedforward=d_model * 4,
            dropout=dropout, activation="gelu", batch_first=True,
        )
        self.encoder = nn.TransformerEncoder(encoder_layer, num_layers=n_layers)
        self.head = nn.Sequential(
            nn.LayerNorm(d_model),
            nn.Linear(d_model, d_model),
            nn.GELU(),
            nn.Linear(d_model, 1),
        )

    def forward(self, x_bins: torch.Tensor) -> torch.Tensor:
        # x_bins: (batch, n_features) int64 in [0, n_bins)
        batch_size = x_bins.shape[0]
        feature_ids = torch.arange(self.n_features, device=x_bins.device).unsqueeze(0)
        tokens = self.value_embedding(x_bins) + self.feature_id_embedding(feature_ids)
        cls = self.cls_token.expand(batch_size, -1, -1)
        tokens = torch.cat([cls, tokens], dim=1)
        encoded = self.encoder(tokens)
        cls_out = encoded[:, 0, :]
        return self.head(cls_out).squeeze(-1)


def build_model(config: dict) -> FTTransformerRegressor:
    return FTTransformerRegressor(
        n_features=config["n_features"],
        n_bins=config.get("n_bins", 7),
        d_model=config.get("d_model", 64),
        n_heads=config.get("n_heads", 4),
        n_layers=config.get("n_layers", 3),
        dropout=config.get("dropout", 0.1),
    )


def save_checkpoint(path, model, config, feature_columns, target_mean, target_std):
    torch.save({
        "model_state": model.state_dict(),
        "config": config,
        "feature_columns": feature_columns,
        "target_mean": target_mean,
        "target_std": target_std,
    }, path)


def load_checkpoint(path, device="cpu"):
    checkpoint = torch.load(path, map_location=device, weights_only=False)
    model = build_model(checkpoint["config"])
    model.load_state_dict(checkpoint["model_state"])
    model.to(device)
    return model, checkpoint

## Load data

In [ ]:
X_train, y_train, X_test = crunch_tools.load_data()
feature_columns = get_feature_columns(X_train)
print(f"{len(feature_columns)} features, {len(X_train):,} rows")

# Features are already quantized into 7 bins (0-6) -- int8 instead of the
# default float64 cuts memory roughly 8x. This is the single biggest lever
# on the ~40GB RAM footprint you mentioned.
X_bins_all = X_train[feature_columns].to_numpy().astype(np.int8)
print(f"Feature matrix memory: {X_bins_all.nbytes / 1e9:.2f} GB (int8)")

## Held-out split & target scaling

Standardizing the target helps training stability (MSE loss on raw skewed
returns can be noisy). We don't need to invert this at inference — Spearman
rank correlation is invariant to any monotonic affine transform, so a
standardized prediction preserves the same ranking as an unstandardized
one. `target_mean`/`target_std` still get saved in the checkpoint so the
CrunchDAO-side fine-tuning uses the exact same scale, rather than each
call computing its own and drifting.

In [ ]:
moons = np.sort(X_train["moon"].unique())
holdout_moons = moons[-50:]
is_holdout = X_train["moon"].isin(holdout_moons).to_numpy()

target = y_train["target"].to_numpy()
target_mean, target_std = float(target[~is_holdout].mean()), float(target[~is_holdout].std())
target_scaled = (target - target_mean) / target_std

X_fit_bins, y_fit = X_bins_all[~is_holdout], target_scaled[~is_holdout]
X_val_bins, y_val_scaled = X_bins_all[is_holdout], target_scaled[is_holdout]
y_val_raw = target[is_holdout]

print(f"train rows: {len(y_fit):,}   holdout rows: {len(y_val_scaled):,}")

## Pretrain

This is the notebook you have real compute for — feel free to sweep
`PRETRAIN_CONFIG` / epochs / learning rate here. Tracks best-epoch Spearman
on the held-out moons and keeps those weights (not necessarily the
last epoch's) for the checkpoint.

In [ ]:
PRETRAIN_CONFIG = {
    "n_bins": 7,
    "d_model": 64,
    "n_heads": 4,
    "n_layers": 3,
    "dropout": 0.1,
}
PRETRAIN_EPOCHS = 30
PRETRAIN_LR = 3e-4
PRETRAIN_BATCH_SIZE = 8192
WEIGHT_DECAY = 1e-5

In [ ]:
# Manual batching over in-memory tensors -- no DataLoader/multiprocessing.
# DataLoader's worker-process machinery exists to parallelize expensive
# per-item work (disk reads, decoding, etc.); we have none of that, the
# whole feature matrix is already a plain array in RAM. Manual slicing is
# simpler AND sidesteps a real gotcha: on macOS/Windows, DataLoader workers
# start via `spawn`, which re-imports __main__ to find classes like a
# custom Dataset -- that doesn't work for classes defined in a notebook
# cell, so num_workers>0 fails there with an AttributeError.
#
# Memory: X_fit_bins/X_val_bins are int8 (1 byte/value). nn.Embedding
# requires int64 indices, but casting the WHOLE array to .long() up front
# turns that back into an 8-bytes/value tensor -- undoing the int8 cast
# and the entire point of it. Instead we keep the full tensors as int8 and
# only cast .long() on each small batch slice, right before it's used.
X_fit_bins_t = torch.from_numpy(X_fit_bins)   # stays int8
y_fit_tensor = torch.from_numpy(y_fit).float()
X_val_bins_t = torch.from_numpy(X_val_bins)   # stays int8
n_train = len(y_fit_tensor)

model = build_model({**PRETRAIN_CONFIG, "n_features": len(feature_columns)}).to(DEVICE)
optimizer = torch.optim.AdamW(model.parameters(), lr=PRETRAIN_LR, weight_decay=WEIGHT_DECAY)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=PRETRAIN_EPOCHS)
loss_fn = nn.MSELoss()


def predict_in_batches(model, X_bins_t, batch_size=PRETRAIN_BATCH_SIZE):
    """Batched inference -- same int8-stays-int8-until-the-last-moment
    reasoning as training: never materialize a whole-dataset int64 copy."""
    model.eval()
    preds = []
    with torch.no_grad():
        for start in range(0, len(X_bins_t), batch_size):
            xb = X_bins_t[start:start + batch_size].long().to(DEVICE)
            preds.append(model(xb).cpu().numpy())
    return np.concatenate(preds)


best_val_spearman = -1.0
best_state = None

for epoch in range(PRETRAIN_EPOCHS):
    model.train()
    total_loss = 0.0
    perm = torch.randperm(n_train)
    for start in range(0, n_train, PRETRAIN_BATCH_SIZE):
        idx = perm[start:start + PRETRAIN_BATCH_SIZE]
        xb = X_fit_bins_t[idx].long().to(DEVICE)   # upcast to int64 only for this batch
        yb = y_fit_tensor[idx].to(DEVICE)
        optimizer.zero_grad()
        pred = model(xb)
        loss = loss_fn(pred, yb)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()
        total_loss += loss.item() * len(yb)
    scheduler.step()

    val_pred = predict_in_batches(model, X_val_bins_t)
    val_spearman = spearman(y_val_raw, val_pred)
    print(f"epoch {epoch + 1}/{PRETRAIN_EPOCHS}  train_loss={total_loss / n_train:.4f}  val_spearman={val_spearman:.4f}")

    if val_spearman > best_val_spearman:
        best_val_spearman = val_spearman
        best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}

model.load_state_dict(best_state)
print(f"\nBest held-out Spearman during pretraining: {best_val_spearman:.4f}")

## Save checkpoint

Upload this file as a resource alongside your CrunchDAO notebook
submission (per CrunchDAO's docs: model files uploaded with a notebook
submission land in the `resources/` directory, the same directory passed
to `train()`/`infer()` as `model_directory_path`). After submitting, check
the run log for a line confirming the file was found — similar to how
`model.joblib` showed up in your earlier successful sklearn-based run —
so you know it warm-started rather than silently cold-starting on CPU.

In [ ]:
save_checkpoint(
    CHECKPOINT_PATH, model,
    config={**PRETRAIN_CONFIG, "n_features": len(feature_columns)},
    feature_columns=feature_columns,
    target_mean=target_mean, target_std=target_std,
)
print(f"Saved checkpoint to {CHECKPOINT_PATH}")
print("Upload this file as a resource alongside competition_transformer.ipynb.")